# 21.4 Attention Scores 🎯

Now we connect **Q, K, and V** into the actual attention mechanism.

Your conceptual flow is exactly:
```
Query × Keys
      ↓
   Scores
      ↓
   Softmax
      ↓
   Weights
      ↓
Weighted Values
```

The important idea is:

> **Attention scores measure how relevant each Key is to the current Query.**

---

## 1. What is an Attention Score?

Suppose we have a Query:

$$
q
$$

and several Keys:

$$
k_1,k_2,\ldots,k_T
$$

We compare the Query with each Key.

The standard comparison is a **dot product**:

$$
\boxed{s_i=q\cdot k_i}
$$

So:
```
             Query
               │
       ┌───────┼────────┐
       ▼       ▼        ▼
      K₁      K₂       K₃
       │       │        │
       ▼       ▼        ▼
     score₁  score₂   score₃
```

A larger score means the Query and Key are more compatible.

---

# 2. Why Dot Product?

The dot product measures how strongly two vectors point in similar directions.

For:

$$
q=[q_1,q_2,\ldots,q_d]
$$

and:

$$
k=[k_1,k_2,\ldots,k_d]
$$

we calculate:

$$
q\cdot k = \sum_{j=1}^{d}q_jk_j
$$

So the attention mechanism gets a numerical measure of **Query–Key compatibility**.

### Intuition
```
Query ─────────► Key
        compare
           │
           ▼
        Score
```

High compatibility → high score.

Low compatibility → low score.

---

# 3. From One Query to All Keys

In real attention, we don't compare the Query with only one Key.

We compare it with **every Key**.

For:

$$
q
$$

and:

$$
K=
\begin{bmatrix}
k_1\\
k_2\\
\vdots\\
k_T
\end{bmatrix}
$$

we obtain:

$$
\boxed{
[q\cdot k_1,\ q\cdot k_2,\ldots,q\cdot k_T]
}
$$

For example:

$$
[1.2,\ 3.8,\ 0.4,\ 2.1]
$$

These are the **raw attention scores**.

---

# 4. Matrix Form

Instead of calculating every dot product individually, Transformers use matrix multiplication.

For:

$$
Q\in\mathbb{R}^{T_q\times d_k}
$$

and:

$$
K\in\mathbb{R}^{T_k\times d_k}
$$

we calculate:

$$
\boxed{QK^T}
$$

giving:

$$
\boxed{
QK^T\in\mathbb{R}^{T_q\times T_k}
}
$$

This matrix contains the Query–Key compatibility scores.

For example:
```
              Keys
           K₁    K₂    K₃    K₄
         ┌──────────────────────
Q₁       │ 1.2   3.8   0.4   2.1
Q₂       │ 2.7   0.8   3.2   1.0
Q₃       │ 0.5   1.9   2.8   4.1
         └──────────────────────
```

Each **row** corresponds to a Query.

Each **column** corresponds to a Key.

This is one of the most important shape concepts in attention.

---

# 5. Why Divide by $\sqrt{d_k}$? 📐

The actual Transformer equation uses **scaled dot-product attention**:

$$
\boxed{
\text{Scores} = \frac{QK^T}{\sqrt{d_k}}
}
$$

The scaling factor is:

$$
\sqrt{d_k}
$$

where $d_k$ is the Key/Query dimension.

### Why?

As $d_k$ becomes larger, dot products can naturally become larger in magnitude.

Very large scores passed into softmax can make the distribution extremely sharp:
```
Without scaling:

[0.001, 0.002, 0.997]
```

This can make gradients less useful.

Scaling keeps the magnitude of the scores under better control.

So:

$$
\boxed{
\text{scaled scores} = \frac{QK^T}{\sqrt{d_k}}
}
$$

Your syllabus explicitly identifies this scaled dot-product formulation as the core attention equation.    syllabus(20261018-070535)

---

# 6. Scores → Softmax

Raw scores aren't probabilities.

Suppose:

$$
\text{scores}=[2.0,\ 1.0,\ 0.1]
$$

We apply softmax:

$$
\boxed{
\alpha_i= \frac{e^{s_i}}{\sum_j e^{s_j}}
}
$$

giving positive values that sum to 1.

For example, approximately:

$$
[0.66,\ 0.24,\ 0.10]
$$

Now we have **attention weights**.

So:
```
Raw scores
[2.0, 1.0, 0.1]
       │
       ▼
    Softmax
       │
       ▼
[0.66, 0.24, 0.10]
       │
       └── sum = 1
```

---

# 7. Why Softmax?

Softmax converts arbitrary scores into a normalized distribution.

Therefore:

$$
\boxed{\alpha_i\geq0}
$$

and:

$$
\boxed{\sum_i\alpha_i=1}
$$

This gives us an intuitive interpretation:

> **How much attention should this Query pay to each Key/Value position?**

For example:

| Token | Score | Attention weight |
| ----- | ----- | ---------------- |
| The   | 0.5   | 0.10             |
| cat   | 3.0   | 0.65             |
| eats  | 1.0   | 0.18             |
| fish  | 0.2   | 0.07             |

The model is primarily attending to **cat**.

---

# 8. Weights → Values 📦

This is where the **Value** from 21.3 comes back.

Suppose:

$$
\alpha=[0.1,\ 0.65,\ 0.18,\ 0.07]
$$

and the corresponding Values are:

$$
v_1,v_2,v_3,v_4
$$

Then:

$$
\boxed{
c=0.1v_1+0.65v_2+0.18v_3+0.07v_4
}
$$

This is the **weighted sum of Values**.

So the attention output contains mostly information from the positions receiving high attention weights.

---

# 9. Complete Pipeline

Now put everything together:
```
                 Q
                 │
                 │
                 ▼
              QKᵀ
                 │
                 ▼
       ┌─────────────────┐
       │ Attention Scores │
       └─────────────────┘
                 │
                 ▼
          Divide by √dₖ
                 │
                 ▼
              Softmax
                 │
                 ▼
       Attention Weights
                 │
                 │ ×
                 ▼
                 V
                 │
                 ▼
       Weighted Sum of V
                 │
                 ▼
          Attention Output
```

Mathematically:

$$
\boxed{
\operatorname{Attention}(Q,K,V)
=
\operatorname{softmax}
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)V
}
$$

This single equation is the mathematical summary of everything we've covered in **21.1–21.4**.

---

# 10. Tiny Numerical Example 🧮

Suppose:

$$
Q=[1,0]
$$

and:

$$
K_1=[1,0]
$$

$$
K_2=[0,1]
$$

Then:

$$
Q\cdot K_1=1
$$

while:

$$
Q\cdot K_2=0
$$

So:

$$
\text{scores}=[1,0]
$$

The Query matches $K_1$ more strongly.

After softmax:

$$
\operatorname{softmax}([1,0]) \approx [0.731,\ 0.269]
$$

Therefore, the corresponding Values are combined approximately as:

$$
\boxed{0.731V_1+0.269V_2}
$$

The output is therefore influenced more strongly by $V_1$.

---

# 11. One Critical Distinction ⚠️

Don't say:

> “The attention score tells us how much information to retrieve.”

More precisely:

### Attention score

$$
QK^T
$$

tells us **how compatible/relevant** a Key is to a Query.

### Attention weight

$$
\operatorname{softmax}
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)
$$

tells us **how strongly each Value contributes**.

So:

$$
\boxed{\text{Score} \neq \text{Weight}}
$$

The weight is obtained **from the score through softmax**.

---

# 12. Q–K–V Final Mental Model 🧠

You can now think about attention as a retrieval process:
```
Q
│
│ "What am I looking for?"
▼
K
│
│ "Which positions match?"
▼
Scores
│
│ "How relevant?"
▼
Softmax
│
│ "How much attention?"
▼
Weights
│
│
▼
V
│
│ "Give me the information."
▼
Attention Output
```

### The golden sentence:

$$
\boxed{
\textbf{Queries and Keys determine attention; Values carry the information being aggregated.}
}
$$

---

## What you should remember 🎯

1. **Attention scores measure Query–Key compatibility.**
2. Scores are calculated using:

$$
QK^T
$$

3. Transformers use scaled scores:

$$
\frac{QK^T}{\sqrt{d_k}}
$$

4. **Softmax converts scores into attention weights.**
5. The weights are applied to **Values**.
6. Final attention output:

$$
\boxed{
\operatorname{softmax}
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)V
}
$$

7. The most important conceptual chain is:

$$
\boxed{
Q\times K
\rightarrow
\text{Scores}
\rightarrow
\text{Softmax}
\rightarrow
\text{Weights}
\rightarrow
V
}
$$

---